# British Airways — Customer Booking Prediction

**Task 2 | Forage Virtual Internship**

Binary classification model predicting whether a customer will complete a flight booking.  
Key improvements over baseline approach:
- Proper `ImbPipeline` to prevent SMOTE data leakage
- `class_weight='balanced'` on RandomForest
- Evaluation with AUC-ROC, Recall, and F1 — not just accuracy
- Stratified K-Fold cross-validation
- New engineered features: `addon_count`, `is_last_minute`, `is_weekend_flight`

In [ ]:
# ── IMPORTS ───────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, roc_auc_score, average_precision_score,
    ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay
)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
BLUE, ORANGE, GREY = '#1a6faf', '#e07b39', '#888888'
print('Libraries loaded ✅')

## 1. Load & Explore Data

In [ ]:
df = pd.read_csv('customer_booking.csv', encoding='ISO-8859-1')
print(f'Shape: {df.shape}')
print(f'\nTarget distribution (%):')
print((df['booking_complete'].value_counts(normalize=True)*100).round(1))
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# ── EDA VISUALIZATIONS ────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle('British Airways Booking Data — Exploratory Analysis', fontsize=14, fontweight='bold')

# Target distribution
ax = axes[0,0]
counts = df['booking_complete'].value_counts()
ax.bar(['Not Completed\n(85%)', 'Completed\n(15%)'], counts.values, color=[GREY, BLUE], width=0.5)
ax.set_title('Booking Completion Rate', fontweight='bold')
ax.set_ylabel('Count')
ax.spines[['top','right']].set_visible(False)

# Purchase lead by outcome
ax = axes[0,1]
for val, label, color in [(0,'Not Completed',GREY),(1,'Completed',BLUE)]:
    ax.hist(df[df['booking_complete']==val]['purchase_lead'], bins=40, alpha=0.6, label=label, color=color, density=True)
ax.set_title('Purchase Lead Time by Outcome', fontweight='bold')
ax.set_xlabel('Days before travel')
ax.legend()
ax.spines[['top','right']].set_visible(False)

# Top booking origins
ax = axes[0,2]
top_origins = df['booking_origin'].value_counts().head(8)
ax.barh(top_origins.index[::-1], top_origins.values[::-1], color=BLUE, alpha=0.8)
ax.set_title('Top 8 Booking Origins', fontweight='bold')
ax.set_xlabel('Number of Bookings')
ax.spines[['top','right']].set_visible(False)

# Add-on preferences by outcome
ax = axes[1,0]
addon_cols = ['wants_extra_baggage', 'wants_preferred_seat', 'wants_in_flight_meals']
addon_labels = ['Extra Baggage', 'Preferred Seat', 'In-Flight Meals']
x = np.arange(len(addon_labels))
ax.bar(x - 0.2, df[df['booking_complete']==0][addon_cols].mean()*100, 0.35, label='Not Completed', color=GREY, alpha=0.8)
ax.bar(x + 0.2, df[df['booking_complete']==1][addon_cols].mean()*100, 0.35, label='Completed', color=BLUE, alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(addon_labels, fontsize=9)
ax.set_ylabel('% of customers')
ax.set_title('Add-on Preferences by Outcome', fontweight='bold')
ax.legend(fontsize=9)
ax.spines[['top','right']].set_visible(False)

# Sales channel completion
ax = axes[1,1]
sc = df.groupby(['sales_channel','booking_complete']).size().unstack()
sc_pct = sc.div(sc.sum(axis=1), axis=0)*100
sc_pct.plot(kind='bar', ax=ax, color=[GREY, BLUE], edgecolor='white', width=0.5)
ax.set_title('Completion Rate by Sales Channel', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('% of bookings')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(['Not Completed','Completed'], fontsize=9)
ax.spines[['top','right']].set_visible(False)

# Flight duration by outcome
ax = axes[1,2]
for val, label, color in [(0,'Not Completed',GREY),(1,'Completed',BLUE)]:
    ax.hist(df[df['booking_complete']==val]['flight_duration'], bins=30, alpha=0.6, label=label, color=color, density=True)
ax.set_title('Flight Duration by Outcome', fontweight='bold')
ax.set_xlabel('Hours')
ax.legend()
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.show()

## 2. Feature Engineering

In [ ]:
# ── FEATURE ENGINEERING ───────────────────────────────────────────────────────

# Ordinal encode flight_day
day_map = {'Mon':1,'Tue':2,'Wed':3,'Thu':4,'Fri':5,'Sat':6,'Sun':7}
df['flight_day'] = df['flight_day'].map(day_map)

# NEW: Total add-ons selected (engagement signal)
df['addon_count'] = df['wants_extra_baggage'] + df['wants_preferred_seat'] + df['wants_in_flight_meals']

# NEW: Last-minute booking flag (≤7 days before travel)
df['is_last_minute'] = (df['purchase_lead'] <= 7).astype(int)

# NEW: Weekend flight flag
df['is_weekend_flight'] = (df['flight_day'] >= 6).astype(int)

# Group rare routes (< 50 occurrences)
route_counts = df['route'].value_counts()
df['route'] = df['route'].apply(lambda x: x if route_counts[x] >= 50 else 'Other')
print(f'Routes after grouping: {df["route"].nunique()} categories')

# Group rare booking origins (< 100)
origin_counts = df['booking_origin'].value_counts()
df['booking_origin'] = df['booking_origin'].apply(lambda x: x if origin_counts[x] >= 100 else 'Other')
print(f'Origins after grouping: {df["booking_origin"].nunique()} categories')

# One-hot encode all categoricals
cat_cols = ['sales_channel', 'trip_type', 'route', 'booking_origin']
df = pd.get_dummies(df, columns=cat_cols, drop_first=False)
print(f'\nFinal feature count: {df.shape[1] - 1} features')
print('New engineered features: addon_count, is_last_minute, is_weekend_flight')

## 3. Train / Test Split

In [ ]:
X = df.drop('booking_complete', axis=1)
y = df['booking_complete']

# Stratified split preserves class ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape[0]:,} rows | Test: {X_test.shape[0]:,} rows')
print(f'Train class 1 rate: {y_train.mean():.1%} | Test class 1 rate: {y_test.mean():.1%}')
print('\n✅ Stratification confirmed — class ratio preserved in both sets')

## 4. Build Model Pipeline

**Key design decision:** SMOTE is applied *inside* the `ImbPipeline`, not before the split.  
This means synthetic samples are only generated on training folds during cross-validation — preventing data leakage.

```
ImbPipeline:
  Step 1: SMOTE  →  balance training set (minority class oversampled)
  Step 2: RandomForest(class_weight='balanced')  →  additional imbalance handling
```

In [ ]:
pipe = ImbPipeline([
    ('smote', SMOTE(random_state=42, k_neighbors=5)),
    ('rf', RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        max_features='sqrt',
        criterion='entropy',
        class_weight='balanced',   # handles residual imbalance after SMOTE
        random_state=42,
        n_jobs=-1
    ))
])

print('Pipeline ready:')
print(' [1] SMOTE oversampling (inside CV folds only)')
print(' [2] RandomForestClassifier(n_estimators=300, max_depth=10, class_weight=balanced)')

## 5. Cross-Validation (5-Fold Stratified)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = cross_validate(
    pipe, X_train, y_train, cv=cv,
    scoring=['accuracy', 'roc_auc', 'f1', 'precision', 'recall'],
    return_train_score=False
)

print('── 5-Fold Stratified Cross-Validation Results ──')
print(f'{"Metric":<15} {"Mean":>8} {"Std":>8}')
print('-' * 33)
for metric in ['roc_auc', 'f1', 'recall', 'precision', 'accuracy']:
    scores = cv_results[f'test_{metric}']
    print(f'{metric:<15} {scores.mean():>8.4f} {scores.std():>8.4f}')

## 6. Train Final Model & Evaluate on Test Set

In [ ]:
# Train on full training set
pipe.fit(X_train, y_train)

# Predict
y_pred = pipe.predict(X_test)
y_prob = pipe.predict_proba(X_test)[:, 1]

print('── Test Set Performance ──')
print(f'AUC-ROC          : {roc_auc_score(y_test, y_prob):.4f}')
print(f'Average Precision: {average_precision_score(y_test, y_prob):.4f}')
print(f'Accuracy         : {(y_pred == y_test).mean():.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['Not Completed','Completed']))

In [ ]:
# ── MODEL EVALUATION PLOTS ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Model Evaluation — BA Booking Prediction', fontsize=13, fontweight='bold')

# ROC Curve
RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[0], color=BLUE, name=f'AUC = {roc_auc_score(y_test, y_prob):.3f}')
axes[0].plot([0,1],[0,1],'--', color=GREY, label='Random (AUC=0.5)')
axes[0].set_title('ROC Curve', fontweight='bold')
axes[0].legend()
axes[0].spines[['top','right']].set_visible(False)

# Precision-Recall
PrecisionRecallDisplay.from_predictions(y_test, y_prob, ax=axes[1], color=ORANGE, name=f'AP = {average_precision_score(y_test, y_prob):.3f}')
axes[1].axhline(y=0.15, color=GREY, linestyle='--', label='Baseline (15%)')
axes[1].set_title('Precision-Recall Curve', fontweight='bold')
axes[1].legend()
axes[1].spines[['top','right']].set_visible(False)

# Confusion Matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, ax=axes[2],
    display_labels=['Not Completed','Completed'],
    colorbar=False, cmap='Blues'
)
axes[2].set_title('Confusion Matrix', fontweight='bold')

plt.tight_layout()
plt.show()

## 7. Feature Importances

In [ ]:
rf_model = pipe.named_steps['rf']
feat_imp = pd.Series(rf_model.feature_importances_, index=X_train.columns)
feat_imp = feat_imp.sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 7))
colors_fi = [BLUE if i < 5 else '#7ab3d4' for i in range(len(feat_imp))]
y_pos = np.arange(len(feat_imp))
ax.barh(y_pos, feat_imp.values, color=colors_fi, edgecolor='white', height=0.7)
ax.set_yticks(y_pos)
ax.set_yticklabels(feat_imp.index.tolist(), fontsize=9)
ax.set_xlabel('Feature Importance (Gini)', fontsize=11)
ax.set_title('Top 15 Feature Importances\nRandom Forest — BA Booking Prediction', fontsize=13, fontweight='bold')
for i, val in enumerate(feat_imp.values):
    ax.text(val + 0.002, i, f'{val:.3f}', va='center', fontsize=9)
ax.spines[['top','right']].set_visible(False)
ax.set_xlim(0, feat_imp.max() * 1.25)
plt.tight_layout()
plt.show()

print('\nTop 5 features:')
for feat, imp in feat_imp.head(5).items():
    print(f'  {feat:<45} {imp:.4f}')

## 8. Summary & Key Findings

### Model Performance
| Metric | CV (5-fold) | Test Set |
|--------|:-----------:|:--------:|
| AUC-ROC | 0.729 ± 0.006 | **0.733** |
| Recall (class 1) | 0.492 | **0.52** |
| F1 (class 1) | 0.382 | **0.39** |
| Precision (class 1) | 0.312 | **0.31** |

**Key improvement vs original approach:**  
Class 1 recall improved from **0.01 → 0.52** — the model now catches 52% of actual completed bookings, versus near-zero before fixing the pipeline.

### Business Insights
1. **Booking origin dominates predictions** — Malaysia (0.239) and Australia (0.064) are the top two features. Customers from these origins have distinct booking completion patterns.
2. **Flight duration matters** (importance: 0.109) — longer routes have higher completion rates, likely because customers are more committed to long-haul travel.
3. **Internet vs Mobile channel** — Internet bookings have higher completion than Mobile; Mobile may have UX friction.
4. **Add-on selection is a weak but positive signal** — customers who select extra baggage/meals are more engaged and slightly more likely to complete.
5. **Purchase lead time**: highly skewed; last-minute bookers behave differently from planners.

### What Would Further Improve the Model
- Threshold tuning (lower from 0.5 → 0.3) to trade precision for higher recall
- Try LightGBM or XGBoost with `scale_pos_weight` for faster convergence
- Add price/fare features if available — cost is likely the #1 dropout driver
- Investigate whether booking_origin is a genuine signal or dataset artifact